In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: Get-Data-sales
position:
  x: 0
  y: 0
description:
  text: Read all data from the sales table.
  hash: 488ca1f8
previewCodeHash: c28997b538a719ed
codegenHash: f9bd80fc93b51b62
config:
  table_source:
    tableName: capgemini_academy.bronze.sales
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "capgemini_academy.bronze.sales"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_0.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_0" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_0.data"])
if "source_0" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_0", "counts": {"data": int(ctx["source_0.data"].count())}}}, raw=True)

In [0]:
"""
id: filter_1
template: filter
templateVersion: 2.0.0
name: Filter-Order_Date
position:
  x: 300
  y: 0
description:
  text: Keep rows where the order date is June 7, 2026; separate the others.
  hash: 9c8b7585
previewCodeHash: 3148a5c3eb726925
codegenHash: d8c91e247e7ed9ff
config:
  condition: Order_Date = TO_DATE('2026-06-07')
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "Order_Date = TO_DATE('2026-06-07')"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_0.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_1.filtered_data"] = out["filtered_data"]
ctx["filter_1.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_1" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_1.filtered_data"])
    display(ctx["filter_1.excluded_data"])
if "filter_1" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "filter_1", "counts": {"filtered_data": int(ctx["filter_1.filtered_data"].count()), "excluded_data": int(ctx["filter_1.excluded_data"].count())}}}, raw=True)

In [0]:
"""
id: select_2
template: transform
templateVersion: 3.0.0
name: Select-Colunas
position:
  x: 600
  y: 0
description:
  text: Only keep columns Order_ID, Customer_ID, and Order_Value.
  hash: 9f7ea6b4
previewCodeHash: d3e032421b4c897f
codegenHash: 1a3c17c6f2a1f062
config:
  mode: select
  edits:
    - column: Order_ID
    - column: Customer_ID
    - column: Order_Value
  ordered: []
input:
  - node: filter_1
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from typing import Any, Dict, List, Optional
from pyspark.sql import functions as F
from pyspark.sql import types as _spark_types
from pyspark.sql.types import (
    BinaryType,
    BooleanType,
    DateType,
    DecimalType,
    FractionalType,
    IntegerType,
    IntegralType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

_TIMESTAMP_TYPES = tuple(
    t
    for t in (TimestampType, getattr(_spark_types, "TimestampNTZType", None))
    if t is not None
)

def _col_ref(name: str):
    if "." in name and not (name.startswith("`") and name.endswith("`")):
        return F.col("`" + name.replace("`", "``") + "`")
    return F.col(name)

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _column_for(item: Dict[str, Any]):
    col = _col_ref(item.get("column", ""))
    alias = item.get("alias")
    if alias:
        col = col.alias(alias)
    return col

def _passthrough_column(name: str, rename_map: Dict[str, Dict[str, Any]]):
    entry = rename_map.get(name)
    col = _col_ref(name)
    if entry and entry.get("alias"):
        col = col.alias(entry["alias"])
    return col

def _type_category(data_type) -> Optional[str]:
    if isinstance(data_type, BooleanType):
        return "boolean"
    if isinstance(data_type, DecimalType):
        return "decimal"
    if isinstance(data_type, IntegralType):
        return "integer"
    if isinstance(data_type, FractionalType):
        return "float"
    if isinstance(data_type, StringType):
        return "string"
    if isinstance(data_type, DateType):
        return "date"
    if isinstance(data_type, _TIMESTAMP_TYPES):
        return "timestamp"
    if isinstance(data_type, BinaryType):
        return "binary"
    return None

_METADATA_SCHEMA = StructType(
    [
        StructField("Name", StringType(), False),
        StructField("Type", StringType(), False),
        StructField("Category", StringType(), True),
        StructField("FieldNumber", IntegerType(), False),
        StructField("IsNumeric", BooleanType(), False),
        StructField("IsInteger", BooleanType(), False),
        StructField("IsFloat", BooleanType(), False),
        StructField("IsString", BooleanType(), False),
        StructField("IsDateOrTime", BooleanType(), False),
        StructField("IsBinary", BooleanType(), False),
    ]
)

def _metadata_row(index: int, field):
    data_type = field.dataType
    category = _type_category(data_type)
    return (
        field.name,
        data_type.simpleString(),
        category,
        index + 1,
        category in ("integer", "float", "decimal"),
        isinstance(data_type, IntegralType),
        category == "float",
        isinstance(data_type, StringType),
        isinstance(data_type, (DateType,) + _TIMESTAMP_TYPES),
        isinstance(data_type, BinaryType),
    )

def _sql_string_literal(value: str) -> str:
    return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"

def _predicate_for(dynamic: Dict[str, Any]) -> str:
    kind = dynamic.get("kind")
    if kind == "byType":
        selected = dynamic.get("types") or []
        if not selected:
            return "false"
        quoted = ", ".join(_sql_string_literal(t) for t in selected)
        return "Category IN (" + quoted + ")"
    if kind == "byName":
        pattern = dynamic.get("namePattern")
        if not pattern or not pattern.get("op"):
            raise ValueError(
                "Select: a byName dynamic rule requires namePattern with an 'op' and 'value'"
            )
        op = pattern.get("op")
        value = pattern.get("value", "")
        case_sensitive = pattern.get("caseSensitive", False) is True
        if op == "regex":
            regex = value if case_sensitive else "(?i)" + value
            return "Name rlike " + _sql_string_literal(regex)
        name_expr = "Name" if case_sensitive else "lower(Name)"
        needle = value if case_sensitive else value.lower()
        escaped = needle.replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_")
        if op == "startsWith":
            like = escaped + "%"
        elif op == "endsWith":
            like = "%" + escaped
        else:
            like = "%" + escaped + "%"
        return name_expr + " like " + _sql_string_literal(like)
    if kind == "byExpression":
        expression = dynamic.get("expression") or ""
        return expression if expression.strip() else "false"
    return "false"

def _resolve_dynamic(df, spark, dynamic: Dict[str, Any]):
    predicate = _predicate_for(dynamic)
    fields = df.schema.fields
    rows = [_metadata_row(i, f) for i, f in enumerate(fields)]
    meta_df = spark.createDataFrame(rows, _METADATA_SCHEMA)
    matched_ordinals = {
        row["FieldNumber"] for row in meta_df.filter(predicate).select("FieldNumber").collect()
    }
    action = dynamic.get("action", "keep")
    return [
        index
        for index in range(len(fields))
        if ((index + 1) in matched_ordinals) != (action == "remove")
    ]

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    dynamic = config.get("dynamic")
    if dynamic:
        kept_indexes = _resolve_dynamic(df, spark, dynamic)
        original_names = [field.name for field in df.schema.fields]
        placeholders = ["_c" + str(i) for i in range(len(original_names))]
        projected = df.toDF(*placeholders).select(*(F.col(placeholders[i]) for i in kept_indexes))
        return {
            "transformed_data": projected.toDF(*(original_names[i] for i in kept_indexes))
        }

    mode = config.get("mode", "passthrough")
    edits: List[Dict[str, Any]] = config.get("edits", [])
    ordered: List[str] = config.get("ordered") or []

    if mode == "select":
        checked_edits = [item for item in edits if _is_checked(item)]
        if not checked_edits:
            return {"transformed_data": df}
        order_index = {name: i for i, name in enumerate(ordered)}
        tail = len(order_index)
        ordered_edits = sorted(
            checked_edits,
            key=lambda item: order_index.get(item.get("column", ""), tail),
        )
        return {"transformed_data": df.select(*(_column_for(item) for item in ordered_edits))}

    unchecked_cols = {
        item.get("column", "")
        for item in edits
        if not _is_checked(item)
    }
    rename_map = {
        item.get("column", ""): item
        for item in edits
        if item.get("alias") and _is_checked(item)
    }
    upstream_cols = list(df.columns)
    upstream_set = set(upstream_cols)

    effective_ordered = ordered if ordered else list(upstream_cols)

    placed = set()
    out = []
    for token in effective_ordered:
        if token in placed or token not in upstream_set:
            continue
        placed.add(token)
        if token in unchecked_cols:
            continue
        out.append(_passthrough_column(token, rename_map))

    for col in upstream_cols:
        if col in placed or col in unchecked_cols:
            continue
        out.append(_passthrough_column(col, rename_map))

    if not out:
        return {"transformed_data": df}
    return {"transformed_data": df.select(*out)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "mode": "select",
    "edits": [
        {
            "column": "Order_ID"
        },
        {
            "column": "Customer_ID"
        },
        {
            "column": "Order_Value"
        }
    ],
    "ordered": []
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_1.filtered_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["select_2.transformed_data"] = out["transformed_data"]
if globals().get("ld_display_outputs", False) or "select_2" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["select_2.transformed_data"])
if "select_2" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "select_2", "counts": {"transformed_data": int(ctx["select_2.transformed_data"].count())}}}, raw=True)

In [0]:
"""
id: aggregate_3
template: aggregate
templateVersion: 2.0.0
name: GroupBy
position:
  x: 900
  y: 0
previewCodeHash: 5539b2e6d73c39f8
codegenHash: 7e9149ee54d999b7
config:
  group_bys:
    - expr: Customer_ID
      type: column
  aggregations:
    - columnExpr:
        expr: Order_Value
        type: column
      fn: SUM
input:
  - node: select_2
    input_port: data
    output_port: transformed_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        elif fn == "COUNT_DISTINCT":
            arg = F.expr(raw_expr) if col_expr.get("type") != "column" else raw_expr
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "Customer_ID",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "Order_Value",
                "type": "column"
            },
            "fn": "SUM",
            "alias": None,
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["select_2.transformed_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["aggregate_3.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False) or "aggregate_3" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["aggregate_3.aggregated_data"])
if "aggregate_3" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "aggregate_3", "counts": {"aggregated_data": int(ctx["aggregate_3.aggregated_data"].count())}}}, raw=True)

In [0]:
"""
id: output_4
template: output
templateVersion: 4.0.0
name: capgemini_academy.silver.customer_revenue
position:
  x: 1200
  y: 0
description:
  text: Save the aggregated data to the specified table, replacing existing data.
  hash: 65bbe258
codegenHash: 8cd4a21ee417df79
config:
  output_type: table
  catalog: capgemini_academy
  schema: silver
  table_name: customer_revenue
  write_mode: overwrite
input:
  - node: aggregate_3
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "capgemini_academy",
    "schema": "silver",
    "table_name": "customer_revenue",
    "write_mode": "overwrite"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["aggregate_3.aggregated_data"]
}
if config["meta_state"]["is_disabled"]:
    out = {"result": spark.sql("SELECT * FROM capgemini_academy.silver.customer_revenue").limit(0)}
else:
    out = run(config, inputs, spark)
ctx["output_4.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_4" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_4.result"])
if "output_4" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_4", "counts": {"result": int(ctx["output_4.result"].count())}}}, raw=True)